In [ ]:
import torch

import time

# import gymnasium

import tqdm
import sys, os
sys.path.append(os.path.dirname(os.path.abspath(os.getcwd())))

# add for visualization
import ipywidgets as widgets
from IPython.display import display, clear_output
import cv2

from src.controller.mppi import MPPI
from src.envs.navigation_2d import Navigation2DEnv

def convert_to_bytes(image):
    _, buffer = cv2.imencode('.jpg', image)
    return buffer.tobytes()

def main(save_mode: bool = False):
    env = Navigation2DEnv()
    video_widget = widgets.Image(format='jpeg')
    display(video_widget)
    # solver
    solver = MPPI(
        horizon=30,
        num_samples=3000,
        dim_state=4,
        dim_control=2,
        dynamics=env.dynamics,
        cost_func=env.cost_function,
        u_min=env.u_min,
        u_max=env.u_max,
        sigmas=torch.tensor([0.5, 0.1]),
        lambda_=1.0,
        auto_lambda=False,
    )

    state = env.reset()
    max_steps = 500
    total_time = 0.0
    step_count = 0
    for i in range(max_steps):
        start = time.time()
        action_seq, state_seq = solver.forward(state=state)
        end = time.time()
        total_time += end - start
        step_count += 1

        current_state = state.detach().clone()
        current_state_batch = current_state.unsqueeze(0).unsqueeze(0)
        state, is_goal_reached = env.step(action_seq[0, :])

        # collision check for predicted trajectory
        is_collisions = env.collision_check(state=state_seq)
        
        # collision check for current robot
        is_robot_collision = env.collision_check(state=current_state_batch)

        top_samples, top_weights = solver.get_top_samples(num_samples=300)

        if save_mode:
            env.render(
                state=current_state,
                predicted_trajectory=state_seq,
                is_collisions=is_collisions,
                top_samples=(top_samples, top_weights),
                is_robot_collision = is_robot_collision,
                mode="rgb_array",
            )
            # progress bar
            if i == 0:
                pbar = tqdm.tqdm(total=max_steps, desc="recording video")
            pbar.update(1)

        else:
            env.render(
                state=current_state,
                predicted_trajectory=state_seq,
                is_collisions=is_collisions,
                is_robot_collision = is_robot_collision,
                top_samples=(top_samples, top_weights),
                mode="human",
            )
        
        # add for visualization
        frame = env.get_current_frame()
        frame_rgb = frame[:, :, :3]  # RGBA → RGB
        frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)  # OpenCV용
        video_widget.value = convert_to_bytes(frame_bgr)
        clear_output(wait=True)
        display(video_widget)
        time.sleep(0.001)  # 100Hz

        if is_goal_reached:
            print("Goal Reached!")
            break

    average_time = total_time / step_count
    print("average solve time: {:.3f} ms".format(average_time * 1000))
    env.close()  # close window and save video if save_mode is True


if __name__ == "__main__":
    main(save_mode = False)
